## Data Cleaning

The goal is to produce a dataset that is consistent, complete, and free of noise before any modeling begins.

### 1.1 Load and Understand the Raw Data

In [2]:
import pandas as pd


df = pd.read_csv("../data/credit_risk_dataset.csv")

print("Shape: ", df.shape) # shape of dataset (rows,columns)

print("First 10 rows: \n", df.head(10).to_string(index=False)) # understand the structure of dataset

print("Types of columns: \n", df.dtypes) # understand the types of each col

print("Statistical summary: \n", df.describe()) # show a statistical summary of dataset

print("Last 10 rows: \n", df.tail(10).to_string(index=False)) # shows the last 10 rows

print("Random 10 rows in dataset: \n",df.sample(10).to_string(index=False)) # show a random 10 rows in dataset

Shape:  (32581, 12)
First 10 rows: 
  person_age  person_income person_home_ownership  person_emp_length loan_intent loan_grade  loan_amnt  loan_int_rate  loan_status  loan_percent_income cb_person_default_on_file  cb_person_cred_hist_length
         22          59000                  RENT              123.0    PERSONAL          D      35000          16.02            1                 0.59                         Y                           3
         21           9600                   OWN                5.0   EDUCATION          B       1000          11.14            0                 0.10                         N                           2
         25           9600              MORTGAGE                1.0     MEDICAL          C       5500          12.87            1                 0.57                         N                           3
         23          65500                  RENT                4.0     MEDICAL          C      35000          15.23            1              

### 1.2 Handle Missing Values

Identify missing values first, then decide on a strategy per column.

In [3]:
print("Missing values per column: \n",df.isnull().sum())

missing_count = df.isnull().sum()
missing_rate  = df.isnull().mean() * 100
 
missing_df = pd.DataFrame({
    "count":   missing_count,
    "rate_%":  missing_rate.round(2),
}).sort_values("rate_%", ascending=False)
 
print(missing_df[missing_df["count"] > 0])


Missing values per column: 
 person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64
                   count  rate_%
loan_int_rate       3116    9.56
person_emp_length    895    2.75


### 1.3 Decide a strategy per column
 
| Missing Rate | Type | Recommended Strategy |
|---|---|---|
| < 5% | Any | Simple imputation (median, mode) |
| 5–30% | MCAR / MAR | Imputation with missing indicator flag |
| > 30% | Any | Consider dropping the column — evaluate carefully | 

### 1.4 Handle Missing Values

In [4]:
from sklearn.impute import KNNImputer
 
# Median imputation (robust to outliers — preferred over mean)
df["loan_int_rate"].fillna(df["loan_int_rate"].median(), inplace=True)

# KNN imputation (best for MAR, uses neighboring rows)
imputer = KNNImputer(n_neighbors=5)
df[["person_emp_length"]] = imputer.fit_transform(df[["person_emp_length"]])

/tmp/ipykernel_68100/2951553083.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["loan_int_rate"].fillna(df["loan_int_rate"].median(), inplace=True)


### 1.5 Remove Impossible Values

These are rows or values that are logically or physically impossible given domain knowledge. They are not outliers — they are errors.

In [5]:
initial_rows = len(df)
 
# Age must be realistic for a loan applicant
df = df[(df["person_age"] >= 18) & (df["person_age"] <= 100)]
 
# Income must be positive
df = df[df["person_income"] > 0]
 
# Employment length cannot exceed age minus legal working age
df = df[df["person_emp_length"] <= (df["person_age"] - 16)]
 
# Employment length cannot be greater than 60 for ex
df = df[df["person_emp_length"] <= 60]

# Loan amount must be positive
df = df[df["loan_amnt"] > 0]
 
# Interest rate must be between 0 and 100
df = df[(df["loan_int_rate"] > 0) & (df["loan_int_rate"] < 100)]
 
removed = initial_rows - len(df)
print(f"Removed {removed} rows with impossible values ({removed / initial_rows:.1%} of data)")

Removed 3781 rows with impossible values (11.6% of data)


### 1.6 Final Audit Before Modeling

In [6]:

# Completeness check
assert df.isnull().sum().sum() == 0, "There are still missing values"
print("No missing values.")

# Range check
assert df["person_age"].between(18, 100).all()
assert df["person_income"].ge(0).all()
assert df["person_emp_length"].le(60).all()
assert df["loan_int_rate"].between(0, 100).all()
assert df["loan_percent_income"].between(0, 10).all()
print("All range checks passed.")

# Shape check
print(f"Final shape: {df.shape}")

### Save the clean dataset
df.to_csv("../data/credit_risk_dataset_cleaned.csv", index=False)


No missing values.
All range checks passed.
Final shape: (28800, 12)
